# TB Notifications Data — Data Cleaning

## 0. Import & Load Data

In [33]:
import pandas as pd

df = pd.read_csv('TB_notifications_LK_edit.csv')

## 1. Exploratory Data Analysis

In [34]:
df.shape

(8492, 25)

In [35]:
df.head()

,country,iso3,iso_numeric,whoregion and year,new_sp,new_sp_m04,new_sp_m514,new_sp_m014,new_sp_m1524,new_sp_m2534,...,new_sp_f04,new_sp_f514,new_sp_f014,new_sp_f1524,new_sp_f2534,new_sp_f3544,new_sp_f4554,new_sp_f5564,new_sp_f65,new_sp_fu
0,Afghanistan,AFG,4,EMR -- 1980,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,4,EMR -- 1981,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,4,EMR -- 1982,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,4,EMR -- 1983,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,4,EMR -- 1984,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8492 entries, 0 to 8491
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             8492 non-null   object 
 1   iso3                8492 non-null   object 
 2   iso_numeric         8492 non-null   int64  
 3   whoregion and year  8492 non-null   object 
 4   new_sp              3902 non-null   float64
 5   new_sp_m04          1071 non-null   float64
 6   new_sp_m514         1082 non-null   float64
 7   new_sp_m014         3168 non-null   float64
 8   new_sp_m1524        3204 non-null   float64
 9   new_sp_m2534        3201 non-null   float64
 10  new_sp_m3544        3214 non-null   float64
 11  new_sp_m4554        3218 non-null   float64
 12  new_sp_m5564        3213 non-null   float64
 13  new_sp_m65          3204 non-null   float64
 14  new_sp_mu           915 non-null    float64
 15  new_sp_f04          1072 non-null   float64
 16  new_sp

In [37]:
df.isnull().sum()

country                  0
iso3                     0
iso_numeric              0
whoregion and year       0
new_sp                4590
new_sp_m04            7421
new_sp_m514           7410
new_sp_m014           5324
new_sp_m1524          5288
new_sp_m2534          5291
new_sp_m3544          5278
new_sp_m4554          5274
new_sp_m5564          5279
new_sp_m65            5288
new_sp_mu             7577
new_sp_f04            7420
new_sp_f514           7407
new_sp_f014           5322
new_sp_f1524          5302
new_sp_f2534          5296
new_sp_f3544          5297
new_sp_f4554          5292
new_sp_f5564          5301
new_sp_f65            5299
new_sp_fu             7579
dtype: int64

In [38]:
# 'whoregion and year' encodes two variables in one column
df['whoregion and year'].unique()[:10]

array(['EMR -- 1980', 'EMR -- 1981', 'EMR -- 1982', 'EMR -- 1983',
       'EMR -- 1984', 'EMR -- 1985', 'EMR -- 1986', 'EMR -- 1987',
       'EMR -- 1988', 'EMR -- 1989'], dtype=object)

In [39]:
# column names encode sex (m/f) and age group — classic untidy structure
# e.g. new_sp_m1524 = male, age 15-24
print(df.columns.tolist())

['country', 'iso3', 'iso_numeric', 'whoregion and year', 'new_sp', 'new_sp_m04', 'new_sp_m514', 'new_sp_m014', 'new_sp_m1524', 'new_sp_m2534', 'new_sp_m3544', 'new_sp_m4554', 'new_sp_m5564', 'new_sp_m65', 'new_sp_mu', 'new_sp_f04', 'new_sp_f514', 'new_sp_f014', 'new_sp_f1524', 'new_sp_f2534', 'new_sp_f3544', 'new_sp_f4554', 'new_sp_f5564', 'new_sp_f65', 'new_sp_fu']


## 2. Data Cleaning

In [40]:
# split 'whoregion and year'
df[['who_region', 'year']] = df['whoregion and year'].str.split(' -- ', expand=True)
df['year'] = df['year'].astype(int)
df = df.drop(columns=['whoregion and year'])

df[['who_region', 'year']].head()

,who_region,year
0,EMR,1980
1,EMR,1981
2,EMR,1982
3,EMR,1983
4,EMR,1984


In [41]:
# melt wide -> long format
# exclude new_sp (total cases) — keep only demographic breakdown columns
demo_cols = [c for c in df.columns if c.startswith('new_sp_')]

tb_long = df.melt(
    id_vars=['country', 'iso3', 'iso_numeric', 'who_region', 'year'],
    value_vars=demo_cols,
    var_name='demographic',
    value_name='cases'
)

In [42]:
# extract sex and age_group from the demographic column name
# e.g. new_sp_m1524 -> sex=Male, age_group=1524
tb_long['sex']       = tb_long['demographic'].str.extract(r'new_sp_([mf])')
tb_long['age_group'] = tb_long['demographic'].str.extract(r'new_sp_[mf](.*)')
tb_long['sex']       = tb_long['sex'].replace({'m': 'Male', 'f': 'Female'})
tb_long = tb_long.drop(columns=['demographic'])

print('sex unique:      ', tb_long['sex'].unique())
print('age_group unique:', tb_long['age_group'].unique())

sex unique:       ['Male' 'Female']
age_group unique: ['04' '514' '014' '1524' '2534' '3544' '4554' '5564' '65' 'u']


## 3. Validation

In [43]:
tb_long.shape

(169840, 8)

In [44]:
tb_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169840 entries, 0 to 169839
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   country      169840 non-null  object 
 1   iso3         169840 non-null  object 
 2   iso_numeric  169840 non-null  int64  
 3   who_region   169840 non-null  object 
 4   year         169840 non-null  int64  
 5   cases        50895 non-null   float64
 6   sex          169840 non-null  object 
 7   age_group    169840 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 10.4+ MB


In [45]:
# cases NaN is expected — structural missingness, not dirty data
tb_long.isnull().sum()

country             0
iso3                0
iso_numeric         0
who_region          0
year                0
cases          118945
sex                 0
age_group           0
dtype: int64

In [46]:
tb_long.head(12)

,country,iso3,iso_numeric,who_region,year,cases,sex,age_group
0,Afghanistan,AFG,4,EMR,1980,NaN,Male,04
1,Afghanistan,AFG,4,EMR,1981,NaN,Male,04
2,Afghanistan,AFG,4,EMR,1982,NaN,Male,04
3,Afghanistan,AFG,4,EMR,1983,NaN,Male,04
4,Afghanistan,AFG,4,EMR,1984,NaN,Male,04
5,Afghanistan,AFG,4,EMR,1985,NaN,Male,04
6,Afghanistan,AFG,4,EMR,1986,NaN,Male,04
7,Afghanistan,AFG,4,EMR,1987,NaN,Male,04
8,Afghanistan,AFG,4,EMR,1988,NaN,Male,04
9,Afghanistan,AFG,4,EMR,1989,NaN,Male,04


## 4. Save

In [47]:
tb_long.to_csv('TB_notifications_cleaned.csv', index=False)
print('Saved: TB_notifications_cleaned.csv')

Saved: TB_notifications_cleaned.csv
